# [CS3780/5780 Sp26] All about Birds — Local Model

**Task**: Multi-class classification of bird species from 64×32 grayscale DCT-spectrogram images.  
**Metric**: Accuracy  
**Classes** (9): Antbird, Bird of prey, Flycatcher, Ground bird, Nocturnal bird, Other non-passerine bird, Parrot, Tanager, Water-associated bird

**AI tool note**: Claude Code (claude-sonnet-4-6) was used to scaffold this notebook — it read the competition PDF, explored the directory structure, checked available conda packages, and generated the initial code. I reviewed and ran all cells.

In [1]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

import time

BASE = Path('All_about_Birds/All_about_Birds')
TRAIN_DIR = BASE / 'train'
TEST_DIR  = BASE / 'test'
SAMPLE_SUB = Path('sample_submission.csv')

print('Train classes:', [d.name for d in sorted(TRAIN_DIR.iterdir()) if d.is_dir()])
print('Test images  :', len(list(TEST_DIR.glob('*.png'))))

Train classes: ['Antbird', 'Bird of prey', 'Flycatcher', 'Ground bird', 'Nocturnal bird', 'Other non-passerine bird', 'Parrot', 'Tanager', 'Water-associated bird']
Test images  : 5454


## 1. Load Training Data

In [2]:
def load_images_from_dir(directory, label=None):
    """Load all PNGs from a directory, return (flat_array, label_list)."""
    imgs, labels = [], []
    for p in sorted(directory.glob('*.png')):
        img = Image.open(p).convert('L')  # grayscale
        arr = np.array(img, dtype=np.float32).flatten() / 255.0
        imgs.append(arr)
        if label is not None:
            labels.append(label)
    return imgs, labels

print('Loading training images...')
t0 = time.time()

X_list, y_list = [], []
for class_dir in sorted(TRAIN_DIR.iterdir()):
    if not class_dir.is_dir():
        continue
    imgs, labels = load_images_from_dir(class_dir, label=class_dir.name)
    X_list.extend(imgs)
    y_list.extend(labels)
    print(f'  {class_dir.name:35s} {len(imgs):5d} images')

X = np.array(X_list)
y = np.array(y_list)
print(f'\nTotal: X={X.shape}, y={y.shape}  ({time.time()-t0:.1f}s)')

Loading training images...
  Antbird                              1586 images
  Bird of prey                         1959 images
  Flycatcher                           5143 images
  Ground bird                           756 images
  Nocturnal bird                       2688 images
  Other non-passerine bird             4144 images
  Parrot                               1175 images
  Tanager                              1750 images
  Water-associated bird                2628 images

Total: X=(21829, 2048), y=(21829,)  (432.8s)


## 2. Load Test Data

In [3]:
print('Loading test images...')
t0 = time.time()

sample_sub = pd.read_csv(SAMPLE_SUB)
test_filenames = sample_sub['file_name'].tolist()

X_test_list = []
for fname in test_filenames:
    img = Image.open(TEST_DIR / fname).convert('L')
    arr = np.array(img, dtype=np.float32).flatten() / 255.0
    X_test_list.append(arr)

X_test = np.array(X_test_list)
print(f'Test: X_test={X_test.shape}  ({time.time()-t0:.1f}s)')

Loading test images...
Test: X_test=(5454, 2048)  (104.7s)


## 3. Encode Labels & Train/Val Split

In [4]:
le = LabelEncoder()
y_enc = le.fit_transform(y)
print('Classes:', list(le.classes_))

X_tr, X_val, y_tr, y_val = train_test_split(
    X, y_enc, test_size=0.15, random_state=42, stratify=y_enc
)
print(f'Train: {X_tr.shape}  Val: {X_val.shape}')

Classes: [np.str_('Antbird'), np.str_('Bird of prey'), np.str_('Flycatcher'), np.str_('Ground bird'), np.str_('Nocturnal bird'), np.str_('Other non-passerine bird'), np.str_('Parrot'), np.str_('Tanager'), np.str_('Water-associated bird')]
Train: (18554, 2048)  Val: (3275, 2048)


## 4. Quick Benchmark — Multiple Classifiers

We use a PCA → classifier pipeline to reduce 2048 dims to a manageable size before fitting.

In [5]:
N_COMPONENTS = 200  # PCA components; covers most variance for 64×32 images

models = {
    'LogReg (C=1)':     LogisticRegression(C=1, max_iter=1000, n_jobs=-1),
    'LogReg (C=10)':    LogisticRegression(C=10, max_iter=1000, n_jobs=-1),
    'SVM-RBF':          SVC(kernel='rbf', C=10, gamma='scale'),
    'ExtraTrees':       ExtraTreesClassifier(n_estimators=300, n_jobs=-1, random_state=42),
    'RandomForest':     RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42),
    'KNN (k=7)':        KNeighborsClassifier(n_neighbors=7, n_jobs=-1),
}

results = []
print(f'{"Model":<25} {"Val Acc":>8}  Time')
print('-' * 45)

# Fit PCA once, reuse across models
scaler = StandardScaler()
pca = PCA(n_components=N_COMPONENTS, random_state=42)

X_tr_s   = scaler.fit_transform(X_tr)
X_val_s  = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

X_tr_pca   = pca.fit_transform(X_tr_s)
X_val_pca  = pca.transform(X_val_s)
X_test_pca = pca.transform(X_test_s)

print(f'PCA explained variance ({N_COMPONENTS} components): {pca.explained_variance_ratio_.sum():.3f}')
print()

best_acc, best_name, best_preds = 0, '', None

for name, clf in models.items():
    t0 = time.time()
    clf.fit(X_tr_pca, y_tr)
    preds = clf.predict(X_val_pca)
    acc = accuracy_score(y_val, preds)
    elapsed = time.time() - t0
    print(f'{name:<25} {acc:>8.4f}  {elapsed:.1f}s')
    results.append({'model': name, 'val_acc': acc})
    if acc > best_acc:
        best_acc, best_name = acc, name
        best_clf = clf

print(f'\nBest: {best_name}  ({best_acc:.4f})')

Model                      Val Acc  Time
---------------------------------------------
PCA explained variance (200 components): 0.883

LogReg (C=1)                0.3258  13.5s
LogReg (C=10)               0.3258  12.8s
SVM-RBF                     0.3377  69.8s
ExtraTrees                  0.3676  2.7s
RandomForest                0.3649  10.6s
KNN (k=7)                   0.2516  0.4s

Best: ExtraTrees  (0.3676)


## 5. Detailed Report for Best Model

In [6]:
val_preds = best_clf.predict(X_val_pca)
print(f'=== {best_name} ===')
print(classification_report(y_val, val_preds, target_names=le.classes_))

=== ExtraTrees ===
                          precision    recall  f1-score   support

                 Antbird       0.45      0.29      0.35       238
            Bird of prey       0.42      0.03      0.06       294
              Flycatcher       0.36      0.78      0.49       772
             Ground bird       0.90      0.08      0.15       113
          Nocturnal bird       0.40      0.23      0.29       403
Other non-passerine bird       0.35      0.50      0.41       622
                  Parrot       0.50      0.01      0.01       176
                 Tanager       0.41      0.21      0.28       263
   Water-associated bird       0.33      0.13      0.19       394

                accuracy                           0.37      3275
               macro avg       0.46      0.25      0.25      3275
            weighted avg       0.40      0.37      0.31      3275



## 6. Retrain Best Model on Full Training Data & Predict Test

In [7]:
print(f'Retraining {best_name} on full training set...')
t0 = time.time()

# Refit scaler + PCA on full training set
X_all_s   = scaler.fit_transform(X)
X_test_s2 = scaler.transform(X_test)

X_all_pca   = pca.fit_transform(X_all_s)
X_test_pca2 = pca.transform(X_test_s2)

best_clf.fit(X_all_pca, y_enc)
test_preds_enc = best_clf.predict(X_test_pca2)
test_preds = le.inverse_transform(test_preds_enc)

print(f'Done in {time.time()-t0:.1f}s')
print('Prediction distribution:')
pd.Series(test_preds).value_counts().sort_index()

Retraining ExtraTrees on full training set...
Done in 5.5s
Prediction distribution:


Antbird                      270
Bird of prey                  49
Flycatcher                  2706
Ground bird                   24
Nocturnal bird               370
Other non-passerine bird    1524
Parrot                        13
Tanager                      215
Water-associated bird        283
Name: count, dtype: int64

## 7. Generate Submission CSV

In [8]:
submission = pd.DataFrame({'file_name': test_filenames, 'label': test_preds})
out_path = 'submission.csv'
submission.to_csv(out_path, index=False)
print(f'Saved {len(submission)} rows to {out_path}')
submission.head(10)

Saved 5454 rows to submission.csv


,file_name,label
0,1.png,Other non-passerine bird
1,2.png,Water-associated bird
2,3.png,Other non-passerine bird
3,4.png,Other non-passerine bird
4,5.png,Flycatcher
5,6.png,Other non-passerine bird
6,7.png,Other non-passerine bird
7,8.png,Flycatcher
8,9.png,Other non-passerine bird
9,10.png,Flycatcher


## 8. (Optional) Try SVM with Larger C on Full Data

SVM-RBF often performs best on PCA-reduced image features. Try tuning C if time allows.

In [11]:
#Quick grid over C for SVM — uncomment to run
for C in [1, 5, 10, 50, 100]:
    clf = SVC(kernel='rbf', C=C, gamma='scale')
    clf.fit(X_tr_pca, y_tr)
    acc = accuracy_score(y_val, clf.predict(X_val_pca))
    print(f'C={C:<5} val_acc={acc:.4f}')

C=1     val_acc=0.3264
C=5     val_acc=0.3392
C=10    val_acc=0.3377
C=50    val_acc=0.3313
C=100   val_acc=0.3221


## 9. (Optional) More PCA Components

Try 300–500 components if the PCA step above doesn't explain enough variance.

In [12]:
for n in [100, 200, 300, 400]:
    pca_n = PCA(n_components=n, random_state=42)
    Xtr_n = pca_n.fit_transform(X_tr_s)
    Xval_n = pca_n.transform(X_val_s)
    clf = SVC(kernel='rbf', C=10, gamma='scale')
    clf.fit(Xtr_n, y_tr)
    acc = accuracy_score(y_val, clf.predict(Xval_n))
    var = pca_n.explained_variance_ratio_.sum()
    print(f'n={n:4d}  var={var:.3f}  acc={acc:.4f}')

n= 100  var=0.840  acc=0.3282
n= 200  var=0.883  acc=0.3377
n= 300  var=0.913  acc=0.3353
n= 400  var=0.938  acc=0.3350
